In [ ]:
import numpy as np
import pandas as pd
import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

### TASK 1

In [ ]:
# Task 1: Train & Evaluate
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names

# Split into 80/20 train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate on the test set
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# Task 1: Serialize and Verify Round-trip
# Save model and target names
joblib.dump(clf, "model.joblib")
joblib.dump(target_names, "target_names.joblib")

# Load them back
loaded_model = joblib.load("model.joblib")
loaded_target_names = joblib.load("target_names.joblib")

# Predict with loaded model
loaded_y_pred = loaded_model.predict(X_test)

# Verify predictions are identical
are_identical = np.array_equal(y_pred, loaded_y_pred)
print(f"Are the predictions identical? {are_identical}")

### TASK 3

In [ ]:
# Task 3: Test the API (Make sure app.py is running on port 5000)

BASE_URL = "http://localhost:5000"

print("--- 1. Health Check ---")
response = requests.get(f"{BASE_URL}/health")
print(f"Status: {response.status_code}, Response: {response.json()}\n")

print("--- 2. Single Prediction (Valid) ---")
valid_payload = {"features": [5.1, 3.5, 1.4, 0.2]}
response = requests.post(f"{BASE_URL}/predict", json=valid_payload)
print(f"Status: {response.status_code}, Response: {response.json()}\n")

print("--- 3. Error Handling (Malformed Requests) ---")
# Missing features key
error_1 = {"wrong_key": [5.1, 3.5, 1.4, 0.2]}
res_err_1 = requests.post(f"{BASE_URL}/predict", json=error_1)
print(f"Error 1 (Missing key) - Status: {res_err_1.status_code}, Response: {res_err_1.json()}")

# Wrong number of features
error_2 = {"features": [5.1, 3.5, 1.4]}
res_err_2 = requests.post(f"{BASE_URL}/predict", json=error_2)
print(f"Error 2 (Wrong length) - Status: {res_err_2.status_code}, Response: {res_err_2.json()}")

# Non-numeric values
error_3 = {"features": ["a", "b", "c", "d"]}
res_err_3 = requests.post(f"{BASE_URL}/predict", json=error_3)
print(f"Error 3 (Non-numeric) - Status: {res_err_3.status_code}, Response: {res_err_3.json()}\n")

print("--- 4. Batch Prediction ---")
# Taking 5 samples from the test set
batch_samples = X_test[:5].tolist()
batch_payload = {"samples": batch_samples}
response = requests.post(f"{BASE_URL}/predict_batch", json=batch_payload)
print(f"Batch Status: {response.status_code}")
import json
print(json.dumps(response.json(), indent=2))

## Task 4: Documentation & Reflection

**1. What would need to change for a production deployment?**
The current setup uses Flask's built-in development server, which handles requests sequentially and is not secure for production. For a real-world deployment, I would replace it with a production-grade WSGI server like Gunicorn. The entire application (code, dependencies, model artifacts) should be containerized using Docker to ensure it runs consistently across different environments. Hardcoded variables (like ports) should be extracted into environment variables. Finally, the service should sit behind a reverse proxy like Nginx to handle HTTPS termination and route traffic securely.

**2. How would you handle model versioning?**
If a new model is trained and deployed, simply overwriting `model.joblib` is dangerous because the new model might perform worse in real-world conditions. I would implement model versioning using a tool like MLflow or DVC, tagging each artifact (e.g., `model_v1`, `model_v2`). During deployment, I would use a "Canary Deployment" or "Shadow Deployment" strategy. For example, 5% of real user traffic could be routed to the new model to verify its performance before completely replacing the old one. This also ensures instant rollbacks are possible.

**3. What monitoring would you add?**
Beyond standard software metrics (CPU/RAM usage, API prediction latency, 400/500 HTTP error rates), a machine learning model requires specific monitoring:
* **Input Data Drift:** Monitoring if the statistical distribution of incoming requests (e.g., petal lengths) significantly differs from the training data.
* **Prediction Drift:** Tracking if the model suddenly starts predicting one class (e.g., `virginica`) overwhelmingly more than usual, which might indicate a problem.

**4. How would the architecture change if the model needed to handle 1,000 requests per second?**
A single Flask app on a single machine cannot handle 1,000 RPS efficiently. The architecture would need to scale horizontally. I would deploy multiple instances of the Docker container managed by an orchestration platform like Kubernetes. A Load Balancer would distribute the incoming traffic across these containers. To further reduce the load, if identical inputs are requested frequently, I would introduce an in-memory caching layer like Redis. Additionally, the model format could be optimized (e.g., converting from scikit-learn/joblib to ONNX format) for faster inference speeds.